#### if you are using databricks make sure to upload the file into volume

In [0]:
path='/Volumes/workspace/default/inputdata/Employee_Dataset_Retail.csv'

In [0]:
df = spark.read.csv(path, header=True, inferSchema=True)
df.createOrReplaceTempView("employee_tbl")
df.columns

In [0]:
%sql
select * from employee_tbl limit 2

#### salary experiments

In [0]:
%sql
select max(Monthly_Income) from employee_tbl

In [0]:
%sql
-- 2nd highest salary using max
select max(Monthly_Income) from employee_tbl where Monthly_Income not in ( select max(Monthly_Income) from employee_tbl)

In [0]:
%sql
-- 2nd highest salary using limit
select employeeMonthly_Income from employee_tbl where Monthly_Income  in ( select monthly_Income from employee_tbl order by 1 desc limit 2)  order by 1 asc limit 1

In [0]:
%sql
-- 2nd highest salary using dense rank
--Employee_ID
select * from (select Employee_ID,Department,Monthly_Income,dense_rank() over(order by Monthly_Income desc) as rank from employee_tbl) where rank = 2

In [0]:
%sql
-- department wise hightest salary
select * from (select Employee_ID,Department,Monthly_Income,rank() over(partition by Department order by Monthly_Income desc) as rnk from employee_tbl) where rnk =2

In [0]:
%sql
-- department wise hightest salary
select * from (select Employee_ID,Department,Monthly_Income,dense_rank() over(partition by Department order by Monthly_Income desc) as rnk from employee_tbl) where rnk =2

In [0]:
%sql
-- 2nd highest salary using dense rank
--Employee_ID
select * from (select Employee_ID,Department,Monthly_Income,dense_rank() over(order by Monthly_Income desc) as rank from employee_tbl) where rank = 2

In [0]:
%sql
SELECT 
    e1.Monthly_Income
FROM 
    employee_tbl e1
JOIN 
    employee_tbl e2
ON 
    e1.Monthly_Income < e2.Monthly_Income
GROUP BY 
    e1.Monthly_Income
HAVING 
    COUNT(DISTINCT e2.Monthly_Income) = 1;


In [0]:
%sql
select a.Monthly_Income from employee_tbl a join employee_tbl b on a.Monthly_Income < b.Monthly_Income group by a.Monthly_Income having COUNT(DISTINCT b.Monthly_Income) = 1

In [0]:
%sql
select Department,max(calculated_diff) from (select *,Hourly_Rate*160 as calcuated_monthlyrate,(Hourly_Rate*160-Monthly_income) as calculated_diff from employee_tbl) group by all

In [0]:
%sql
select Employment_Type,department,max(Monthly_Income) over(partition by department,Employment_Type order by Monthly_Income) from employee_tbl

#### top 5 highest‑paid employees in each department

In [0]:
%sql
select * from (select Employee_Name,Department,Monthly_Income,row_number() over(partition by Department order by Monthly_Income desc) as rnk from employee_tbl) where rnk <=5

In [0]:
%sql
select Department,max(Monthly_Income) from employee_tbl group by all

#### average income by education level and gender

In [0]:
%sql
select Employee_Name,Education_Level,Gender,avg(Monthly_Income)
 from employee_tbl group by all

#### employees whose performance score is above the department average

In [0]:
%sql
select Employee_Name,Department,Performance_Score,Last_3_Year_Rating,Annual_HR_Feedback_Satisfaction_Score from employee_tbl

In [0]:
%sql
select distinct Department,Annual_HR_Feedback_Satisfaction_Score,avg from (select Employee_Name,Department,Annual_HR_Feedback_Satisfaction_Score,
       avg(Annual_HR_Feedback_Satisfaction_Score) over(partition by Department) as avg
from employee_tbl) where Annual_HR_Feedback_Satisfaction_Score > avg

#### Find the attrition rate by department

In [0]:
%sql
select Department,round(avg(att_status),2) from (select Department,case when Attrition_Status = 'No' then 0 else 1 end as att_status from employee_tbl) group by all order by 2 desc

#### most common attrition reason for each department

In [0]:
%sql
select distinct Department,Attrition_Reason,count(*)as cnt,rank() over (partition by Department order by count(*) desc) as rnk from employee_tbl where Attrition_status = 'Yes' 
group by Department,Attrition_Reason

In [0]:
%sql
select * from (select distinct Department,Attrition_Reason,count(*),rank() over (partition by Department order by count(*) desc) as rnk from employee_tbl where Attrition_status = 'Yes' 
group by Department,Attrition_Reason) where rnk = 1

In [0]:
%sql
select * from employee_tbl

#### income is above the 90th percentile in their department


In [0]:
%sql
select distinct Department,Monthly_Income,avg from
(select Department,Monthly_Income,avg(Monthly_Income) over (partition by Department) as avg from employee_tbl group by all) where Monthly_Income > avg*0.9

#### year‑over‑year attrition change for each department


In [0]:
%sql
select Department,sum(att_year-yoj) from (select Department,year(Date_of_Joining) as yoj,right(Attrition_Date,4) as att_year from employee_tbl where Attrition_Status = 'Yes') group by all order by 2 desc